# Import Library

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.losses import Huber
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from datetime import datetime

# Data Understanding

In [2]:
df = pd.read_csv('/content/DataProduksi.csv')
df

,Tanggal,Minyak Sawit,Karet Kering,Teh
0,Jan-09,1427.10,50.00,8.80
1,Feb-09,1188.00,45.50,7.90
2,Mar-09,1346.70,40.10,8.50
3,Apr-09,1193.50,38.80,9.30
4,May-09,1239.50,47.20,10.30
...,...,...,...,...
115,Aug-18,2075.95,35.32,5.99
116,Sep-18,2093.69,49.37,6.78
117,Oct-18,2075.70,51.21,8.12
118,Nov-18,1931.94,48.77,8.48


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Tanggal       120 non-null    object 
 1   Minyak Sawit  120 non-null    float64
 2   Karet Kering  120 non-null    float64
 3   Teh           120 non-null    float64
dtypes: float64(3), object(1)
memory usage: 3.9+ KB


# Preprocessing

## Change date format

In [4]:
df['Tanggal'] = pd.to_datetime(df['Tanggal'], format='%b-%y').dt.strftime('%Y-%m')
df.head()

,Tanggal,Minyak Sawit,Karet Kering,Teh
0,2009-01,1427.1,50.0,8.8
1,2009-02,1188.0,45.5,7.9
2,2009-03,1346.7,40.1,8.5
3,2009-04,1193.5,38.8,9.3
4,2009-05,1239.5,47.2,10.3


## Normalization

In [5]:
scaler = MinMaxScaler()
columns = ['Minyak Sawit', 'Karet Kering', 'Teh']
df[columns] = scaler.fit_transform(df[columns])
df.head()

,Tanggal,Minyak Sawit,Karet Kering,Teh
0,2009-01,0.434536,0.351266,0.655963
1,2009-02,0.248938,0.268072,0.449541
2,2009-03,0.372127,0.168238,0.587156
3,2009-04,0.253208,0.144204,0.770642
4,2009-05,0.288915,0.299501,1.000000


## Separating DataFrame Columns into Separate Variables

In [6]:
dates = df['Tanggal'].values
minyak_sawit = df['Minyak Sawit'].values
karet_kering = df['Karet Kering'].values
teh = df['Teh'].values

## Windowed Dataset

In [7]:
def windowed_dataset(series, window_size, batch_size, shuffle_buffer):
    series = tf.expand_dims(series, axis=-1)
    ds = tf.data.Dataset.from_tensor_slices(series)
    ds = ds.window(window_size + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda w: w.batch(window_size + 1))
    ds = ds.shuffle(shuffle_buffer)
    ds = ds.map(lambda w: (w[:-1], w[-1:]))
    return ds.batch(batch_size).prefetch(1)

## Split Dataset

In [8]:
def split_data(data):
    train_size = int(len(data) * 0.8)
    train_data = data[:train_size]
    test_data = data[train_size:]
    return train_data, test_data

# Generate Output

## Generate MAE & R2

In [9]:
def generate_mae_r2(X_test, y_pred):
  print('Mean Absolute Error:', mean_absolute_error(X_test, y_pred))
  print('R2 Score:', r2_score(X_test, y_pred))

## Generate Plot for Data test and Prediction

In [10]:
  def generate_plot(X_test, y_pred):
    plt.figure(figsize=(10, 6))
    plt.plot(X_test, label='Data Test')
    plt.plot(y_pred, label='Data Prediksi')
    plt.title('Data Test vs Data Prediksi')
    plt.xlabel('Bulan')
    plt.ylabel('Nilai Produksi')
    plt.legend()
    plt.show()

# LSTM model

In [16]:
def model_lstm(data, attribute_name):
    train_data, test_data = split_data(data)
    window_size = 12
    batch_size = 32
    shuffle_buffer_size = 1000

    train_set = windowed_dataset(train_data, window_size, batch_size, shuffle_buffer_size)
    test_set = windowed_dataset(test_data, window_size, batch_size, shuffle_buffer_size)

    model = Sequential([
        LSTM(128, return_sequences=True),
        Dropout(0.2),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])

    model.compile(loss=Huber(), optimizer=SGD(learning_rate=1e-8, momentum=0.9), metrics=['mae'])
    model.fit(train_set, epochs=1, verbose=0)

    forecast = []
    for time in range(len(test_data) - window_size):
        forecast.append(model.predict(test_data[time:time + window_size][np.newaxis]))

    forecast = forecast[window_size:]
    results = np.array(forecast)

    X_test = test_data[window_size:]
    y_pred = results

    generate_mae_r2(X_test, y_pred)
    generate_plot(X_test, y_pred)

    return model

# SVR Model

In [12]:
def model_svr(data, attribute_name):
  X_train, X_test = split_data(data)
  model = make_pipeline(MinMaxScaler(), SVR(C=1.0, epsilon=0.2))
  model.fit(X_train.reshape(-1,1), X_train.reshape(-1,1))
  y_pred = model.predict(X_test.reshape(-1,1))

  generate_mae_r2(X_test, y_pred)
  generate_plot(X_test, y_pred)

# LSTM Evaluation

## Minyak Sawit

In [17]:
model_lstm(minyak_sawit, 'Minyak Sawit')

1/1 [==============================] - 0s 24ms/step


ValueError: ignored

## Karet Kering

In [ ]:
model_lstm(karet_kering, 'Karet Kering')

## Teh

In [ ]:
model_lstm(teh, 'Teh')

# SVR Evaluation

## Minyak Sawit

In [ ]:
model_svr(minyak_sawit, 'Minyak Sawit')

## Karet Kering

In [ ]:
model_svr(karet_kering, 'Karet Kering')

## Teh

In [ ]:
model_svr(teh, 'Teh')